In [14]:
!pip install transformers

In [15]:
from transformers import pipeline
import torch

In [3]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [21]:
#Load the model pipeline
generator = pipeline(task="text-generation", model="distilgpt2")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use mps:0


In [22]:
Prompt = "Michigan city in Indiana is famous for "
generated_text = generator(Prompt, max_length = 150, pad_token_id=generator.tokenizer.eos_token_id, truncation = True)
print(generated_text[0]['generated_text'])

Michigan city in Indiana is famous for  an American immigrant living in Pennsylvania. He is an American and has raised a daughter in Pennsylvania.

The last time someone met an American was when a male named Daphne (daphne) first moved north of the capital and said "Daphne" must have been born while at the same time, and Daphne was on the list. But in order to learn this, he needed a mother and a daughter for him for three weeks that day.
A father called his own and he went to the school, where he taught English. But what would Daphne be known to be?
The father mentioned that a teacher had asked him to become a teacher and, because


In [28]:
pip install datasets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.9/30.9 MB 29.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 27.9 MB/s eta 0:00:0000:010:01
  Attempting uninstall: fsspec━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/17 [pyarrow]
    Found existing installation: fsspec 2025.3.2━━━━━━━━━━━━━━  3/17 [pyarrow]
    Uninstalling fsspec-2025.3.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/17 [pyarrow]
      Successfully uninstalled fsspec-2025.3.2━━━━━━━━━━━━━━━━  3/17 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [datasets]/17 [datasets]ess]
Note: you may need to restart the kernel to use updated packages.


Preparing for fine tuning

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

/Users/jahnavi/Documents/llm-finetune/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data = load_dataset("imdb", split='train')
train_data = train_data.shard(num_shards=4, index=0)
test_data = load_dataset("imdb", split='test')
test_data = test_data.shard(num_shards=4, index=0)

In [3]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

#Tokenize the data
tokenized_training_data = tokenizer(train_data["text"], return_tensors='pt', padding=True, truncation=True, max_length = 64)
tokenized_test_data = tokenizer(test_data["text"], return_tensors='pt', padding=True, truncation=True, max_length = 64)

#Tokenizing the batch


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
print(tokenized_training_data)
print(tokenized_test_data)

{'input_ids': tensor([[  101,  1045, 12524,  ...,  1000,  1045,   102],
        [  101,  1000,  1045,  ...,  1005,  1056,   102],
        [  101,  2065,  2069,  ..., 11795,  3085,   102],
        ...,
        [  101, 17012,  1010,  ...,  2668,  8631,   102],
        [  101,  2066,  2087,  ...,  2781,  1012,   102],
        [  101,  1045,  2079,  ...,  1997,  2014,   102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])}
{'input_ids': tensor([[  101,  1045,  2293,  ...,  1996,  2434,   102],
        [  101,  4276,  1996,  ...,  2498,  2008,   102],
        [  101,  20

In [12]:
def tokenizer_function(data):
    return tokenizer(
        data["text"],
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

tokenized_training_data = train_data.map(tokenizer_function, batched=True)

Map: 100%|██████████| 6250/6250 [00:00<00:00, 7903.15 examples/s]


In [13]:
training_args = TrainingArguments(
    output_dir="./finetuned",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset=tokenized_training_data,
    eval_dataset= tokenized_test_data,
    processing_class = tokenizer
)

trainer.train()

/Users/jahnavi/Documents/llm-finetune/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.007900
1000,0.000000
1500,0.000000
2000,0.000000


/Users/jahnavi/Documents/llm-finetune/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/jahnavi/Documents/llm-finetune/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=2346, training_loss=0.0017143702519563711, metrics={'train_runtime': 474.8852, 'train_samples_per_second': 39.483, 'train_steps_per_second': 4.94, 'total_flos': 616666536000000.0, 'train_loss': 0.0017143702519563711, 'epoch': 3.0})

In [17]:
# Force everything to run on CPU
device = torch.device("cpu")
model = model.to(device)

In [23]:
new_data = ["soooo gooddddd", "SOOOO goodddddd", "Haha! very good, feels good"]
new_input = tokenizer(new_data, return_tensors='pt', padding=True, truncation=True, max_length=64)
with torch.no_grad():
    outputs = model(**new_input)
predicted_labels = torch.argmax(outputs.logits, dim=1).tolist()
label_map = {0: "Negative", 1:"Positive"}
for i, predicted_label in enumerate(predicted_labels):
    sentiment = label_map[predicted_label]
    print(f"\nInput Test{i+1}: {new_data[i]}")
    print(f"Predicted Label: {sentiment}")



Input Test1: soooo gooddddd
Predicted Label: Negative

Input Test2: SOOOO goodddddd
Predicted Label: Negative

Input Test3: Haha! very good, feels good
Predicted Label: Negative
